# How the §14 secondary analyses are derived

The primary analysis (`primary_outcome.ipynb`) answers a question about patients who could plausibly
have received either treatment, at centres using both. Two questions it cannot answer:

- **§14a** — What if *every* IVT-eligible patient, at *all* centres — including one that never gives
  IVT — had been bridged, versus none? (An **average treatment effect** in the all-centre eligible population.)
- **§14b** — What would a *policy* of "bridge everyone eligible, direct EVT for the contraindicated"
  deliver, against direct EVT for all? (An operational, not biological, question.)

Neither can use propensity weighting: a centre with no bridged patient has no propensity contrast.
Both use **standardisation** (g-computation) instead:

```
fit one ordinal model  ─►  predict every patient's mRS distribution under each regime  ─►  average
```

Section references `[§n]` point to `statistical_analysis_plan.md`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent))     # the analysis modules live one directory up

import config as C
import data, derive, eligibility, cohort, propensity, outcome, bootstrap, model, sensitivity, standardise, policy

pd.set_option("display.precision", 3)
plt.rcParams["figure.dpi"] = 110

ARM = C.TREATMENT_LABELS        # {0: "EVT alone", 1: "bridging"}
COLOURS = {0: "#7f8c8d", 1: "#2980b9"}

## 1. Populations

Three nested populations; the primary uses the innermost.

```
all patients, all centres (123)  ───────────────────────────────  §14b
   └─ IVT-eligible, all centres (104)  ─────────────────────────  §14a
        └─ IVT-eligible, centres using both strategies (93)  ───  primary [§3]
```

In [ ]:
data_path = Path('/Users/jk/stroke_datasets/Excel_bridging_EXTEND_paper_HUG_CHUV_LUGANO_USZ_def_v7_july26_with_abs_contra_indication.xlsx')
N_RECORDS_EXPECTED = 126
DATA_SHA256 = "54934fbb2ae22647a9c0a2cbaff7ac425ed8948bcaeabe36d69aca00df657371"

data.WORKBOOK = data.Source(
    data_path,
    DATA_SHA256, N_RECORDS_EXPECTED,
    "v7_july26_with_abs_contra_indication")

In [ ]:
df, audit = data.load(data.WORKBOOK)
df = derive.derive(df, audit)
classified = eligibility.classify(df, audit)        # every patient, labelled eligible / contraindicated

pop_a = standardise.population(classified, audit)   # §14a: eligible, all centres
pop_b = policy.population(classified, audit)        # §14b: everyone, all centres

primary_cohort = cohort.build(classified, audit)
populations = {"workbook": classified, "§14b population": pop_b, "§14a population": pop_a, "primary cohort": primary_cohort}
flow = pd.DataFrame({"patients": {k: len(v) for k, v in populations.items()},
                     "bridged": {k: int(v[C.TREATMENT].sum()) for k, v in populations.items()}})
display(flow)

pop_a.groupby("center")[C.TREATMENT].agg(["count", "sum"]).rename(columns={"count": "patients", "sum": "bridged"})

One centre bridged nobody. Under weighting it had to be excluded; under standardisation its patients
contribute their covariates and their observed EVT-alone outcomes, and their *bridging* outcome is
**predicted** from what the model learned at the other centres.

## 2. §14a — the model

One proportional-odds model on all eligible patients, treatment plus the confounders `[§6]`, but
**without `center`**:

```
logit P(mRS ≤ k | A, X) = α_k + β·A + γᵀX
```

Centre is left out on purpose. The analysis assumes that, given `X`, outcome does not depend on the
centre — that is what lets the never-IVT centre borrow the treatment effect. Including centre would
contradict the assumption the analysis runs on. This assumption is **not testable** from the data.

`exp(β)` here is a *conditional* odds ratio (given `X`). It is a model parameter, not the result.

In [ ]:
std = standardise.all_centre(pop_a, audit)

coef = pd.Series(std.fit.beta, index=std.fit.columns, name="coefficient")
display(coef.to_frame())
print(f"conditional odds ratio exp(β) = {std.conditional_odds_ratio:.3f}   (model parameter only)")

## 3. §14a — standardisation

Every patient is duplicated: once with `A = 1`, once with `A = 0`. The model predicts the full mRS
distribution `P(Y = j)`, `j = 0…6`, for both copies. Averaging each over all 104 patients gives two
standardised distributions — "everyone bridged" and "nobody bridged" — for the *same* people.

From those two distributions come the reportable quantities: cumulative `RD_k`, the mRS 0–2
difference (`= RD_2`), and the mortality difference (`= −RD_5`).

In [ ]:
def stacked(ax, dist_by_key, names, title):
    # Stacked horizontal bars of P(mRS = j) per regime.
    cmap = plt.get_cmap("RdYlGn_r", len(C.MRS_LEVELS))
    for i, (key, name) in enumerate(names.items()):
        left = 0.0
        for j in C.MRS_LEVELS:
            p = dist_by_key[key][j]
            ax.barh(i, p, left=left, color=cmap(j), edgecolor="white")
            if p > 0.04:
                ax.text(left + p / 2, i, str(j), ha="center", va="center", fontsize=8)
            left += p
    ax.set_yticks(range(len(names)), list(names.values()))
    ax.set_xlim(0, 1); ax.set_title(title)


fig, ax = plt.subplots(figsize=(8, 2.2))
stacked(ax, std.distribution, {1: "all bridged", 0: "none bridged"}, "§14a standardised mRS, all eligible (n=104)")
plt.show()

pd.DataFrame({"RD_k": std.rd,
              "P(mRS ≤ k) none bridged": {k: std.cumulative[0][k] for k in C.MRS_THRESHOLDS},
              "P(mRS ≤ k) all bridged": {k: std.cumulative[1][k] for k in C.MRS_THRESHOLDS}}).rename_axis("k")

## 4. §14a — is the model extrapolating?

Standardisation never produces infinite weights, but it *can* predict for patients unlike any who were
actually bridged. The check: take the range of every continuous covariate among bridged patients (the
"treated support box") and count never-IVT-centre patients outside it. Sensitivity: average only over
patients inside the box (the fit is unchanged).

In [ ]:
sup = standardise.support(pop_a, audit)

box = pd.DataFrame(sup.box, index=["treated min", "treated max"]).T
display(box)
print(f"never-IVT centre(s): {sup.never_ivt}; {sup.n_never_ivt_outside} of {sup.n_never_ivt} patients outside the box")
print("outside the box, by centre:", sup.outside_by_centre)

std_sup = standardise.all_centre(pop_a, audit, over=sup.inside)
pd.DataFrame({"all eligible": std.rd, "inside treated support": std_sup.rd}).rename_axis("k")

## 5. §14a — sensitivity: a random centre intercept

A hierarchical variant lets each centre have its own baseline (`b_c ~ N(0, σ²)`) while keeping one
common treatment effect. If centres truly do not differ given `X`, `σ ≈ 0` and this collapses to the
pooled model. Four centres give a fragile `σ`; this is a check on the pooled model, not a replacement.

In [ ]:
hier = standardise.hierarchical(pop_a, audit)

print(f"between-centre SD σ = {hier.sigma:.3f}" + ("  (at the floor)" if hier.at_floor else ""))
display(pd.Series(hier.intercepts, name="centre intercept b_c").to_frame())
pd.DataFrame({"pooled": std.rd, "random intercept": hier.standardisation.rd}).rename_axis("k")

## 6. §14b — the policy contrast

Same machinery, all 123 patients, with two changes:

1. The model adds a **contraindication indicator** as a covariate. Without it, "no IVT" in a
   contraindicated patient would double as a marker of their (worse) prognosis.
2. Treatment under the active regime is **not a constant**: eligible → bridged, contraindicated →
   direct EVT. Under the comparator regime everyone gets direct EVT.

A contraindicated patient is treated identically under both regimes, so contributes exactly 0 to every
risk difference. The policy effect is therefore the eligible-patient contrast *diluted* by the eligible
share of the cohort — both factors are reported.

In [ ]:
pol = policy.contrast(pop_b, audit)

print(f"eligible share = {pol.n_eligible}/{pol.n_average} = {pol.share_eligible:.3f}")
fig, ax = plt.subplots(figsize=(8, 2.2))
stacked(ax, pol.distribution, {"active": "bridge eligible", "comparator": "direct EVT for all"},
        "§14b standardised mRS, all patients (n=123)")
plt.show()

pd.DataFrame({"policy RD_k": pol.rd,
              "eligible-only RD_k (this model)": pol.eligible_rd,
              "eligible RD_k × share": {k: v * pol.share_eligible for k, v in pol.eligible_rd.items()}}).rename_axis("k")

## 7. Uncertainty

Bootstrap as for the primary `[§10]`: 2000 resamples stratified by centre; in each, refit the ordinal
model, re-predict every resampled patient under both regimes, average, recompute. Percentile intervals.
**No p-values** are prescribed for §14 — these are estimation-only analyses.

These two cells take about five minutes together.

In [ ]:
boot_a = standardise.inference(pop_a, audit)     # covers pooled, treated-support and hierarchical arms
boot_b = policy.inference(pop_b, audit)


def iv(boot, key):
    i = boot.intervals[key]
    return i.lo, i.hi, i.n_draws


rows = []
for k in C.MRS_THRESHOLDS:
    rows.append((f"RD_{k}", std.rd[k], *iv(boot_a, f"rd_{k}")[:2],
                 std_sup.rd[k], *iv(boot_a, f"support.rd_{k}")[:2],
                 hier.standardisation.rd[k], *iv(boot_a, f"hier.rd_{k}")[:2],
                 pol.rd[k], *iv(boot_b, f"rd_{k}")[:2]))
rows.append(("mortality", std.mortality, *iv(boot_a, "mortality")[:2],
             std_sup.mortality, *iv(boot_a, "support.mortality")[:2],
             hier.standardisation.mortality, *iv(boot_a, "hier.mortality")[:2],
             pol.mortality, *iv(boot_b, "mortality")[:2]))

cols = pd.MultiIndex.from_product([["§14a all eligible", "§14a treated support", "§14a random intercept", "§14b policy"],
                                   ["est", "lo", "hi"]])
table = pd.DataFrame([r[1:] for r in rows], index=[r[0] for r in rows], columns=cols)
table

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
ks = np.array(C.MRS_THRESHOLDS, dtype=float)
for off, (name, rd, key_fmt, boot) in enumerate([
        ("§14a all eligible", std.rd, "rd_{}", boot_a),
        ("§14a treated support", std_sup.rd, "support.rd_{}", boot_a),
        ("§14a random intercept", hier.standardisation.rd, "hier.rd_{}", boot_a),
        ("§14b policy", pol.rd, "rd_{}", boot_b)]):
    lo = np.array([boot.intervals[key_fmt.format(k)].lo for k in C.MRS_THRESHOLDS])
    hi = np.array([boot.intervals[key_fmt.format(k)].hi for k in C.MRS_THRESHOLDS])
    est = np.array([rd[k] for k in C.MRS_THRESHOLDS])
    x = ks + (off - 1.5) * 0.15
    ax.errorbar(x, est, yerr=[est - lo, hi - est], fmt="o", capsize=2, label=name)
ax.axhline(0, color="red", ls="--")
ax.set_xticks(ks, [f"mRS ≤ {int(k)}" for k in ks]); ax.set_ylabel("RD_k"); ax.legend(fontsize=8)
ax.set_title("Cumulative risk differences with 95% percentile CIs")
plt.show()

## 8. Reading the results

- **Three different questions, three different populations.** Primary (ATO, 92 patients, 3 centres),
  §14a (ATE, 104 eligible patients, 4 centres), §14b (policy, 123 patients). Differences between them
  are not "the same effect with more data" — they are different estimands `[§14]`.
- **§14a rests on an untestable assumption**: given the confounders, centre does not affect outcome, and
  the treatment effect transports to a centre where it was never observed. The support check and the
  random-intercept sensitivity probe it; they cannot verify it.
- **§14b is operational.** It says what a bridging policy would have delivered in this cohort, not what
  IVT does biologically. Its effect is the eligible contrast scaled by the eligible share.
- **`exp(β)` from these models is conditional** and appears only as a model parameter; the reported
  effects are the standardised risk differences.
- **Intervals only, no p-values**, by design.